# Task 2: Predicting severity metrics
Run this notebook to predict the severity (UPDRSM). 

Structure of this notebook: 
1. Set up
2. Model 1: Multiple Linear Regression (from `src/models/linreg.py`) and Performance 
3. Model 2: Random Forest (from `src/models/rf.py`) and Performance
4. Comparing Model 1 and Model 2
5. Model 3: 1DCNN (from `src/models/cnn1d.py`) and Performance
6. Comparing across all 3 models 

## 1. Set up and data preparation

In [ ]:
# RUN CODE: execute this cell first, once per new kernel.
from pathlib import Path  
import sys  
import numpy as np  
import pandas as pd 
import joblib  # Save fitted classical models for reproducibility.
import matplotlib.pyplot as plt

# CHECK PATH: start Jupyter in repo root OR in repo/notebooks.
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src').exists(), f'Cannot locate repo src/ folder from {Path.cwd()}' # checks that models exist in src
sys.path.insert(0, str(ROOT))

FEATURES_PATH = ROOT / 'data' / 'features.csv' # to check if correct 
METADATA_PATH = ROOT / 'data' / 'others' / 'subject_metadata.xls' # to check if correct

# saving output in folder
OUT = ROOT / 'results' / 'task2' 
(OUT / 'predictions').mkdir(parents=True, exist_ok=True)
(OUT / 'weights').mkdir(parents=True, exist_ok=True)
(OUT / 'figures').mkdir(parents=True, exist_ok=True)

# checkpoints
print('Repo root:', ROOT)
print('Features exist:', FEATURES_PATH.exists())
print('Metadata exists:', METADATA_PATH.exists())
print('Output folder exists:', OUT.exists())

In [ ]:
# check metadata and print 
metadata = pd.read_excel(METADATA_PATH)  
print('Metadata dimensions:', metadata.shape)
print('Metadata columns:', metadata.columns.tolist())
display(metadata.head(5))  

assert {'Study', 'Group', 'Subjnum', 'UPDRSM'}.issubset(metadata.columns) # check for required rows

# check features and print
features = pd.read_csv(FEATURES_PATH)
print('Feature table dimensions:', features.shape)
print('Feature columns:', features.columns.tolist())
display(features.head(5))

### Merge metadata with features
The metadata contains our target UPDRSM, whereas features contain the gait characteristics used for prediction. We merge the dataframes. 

NOTE: to check if need to conduct checks on the metadata and features here, might not need to but already added in case. 

In [ ]:
metadata = metadata.copy()  # make copy to avoid editing the file accidentally 

# check metadata dataframe, standardise group labels
metadata['Group'] = metadata['Group'].astype(str).str.strip().replace({'PD': 'Pt', 'CO': 'Co'})
metadata['subject'] = (metadata['Study'].astype(str).str.strip()
                       + metadata['Group'].astype(str).str.strip()
                       + pd.to_numeric(metadata['Subjnum'], errors='raise').astype(int).astype(str).str.zfill(2))
assert metadata['subject'].is_unique, 'Metadata contains duplicate subject IDs.' # check for duplicates 

# check features dataframe 
if 'subject' not in features.columns:
    candidates = [c for c in ['file', 'filename', 'walk_file'] if c in features.columns]
    assert candidates, 'Features need a subject column or a file/filename/walk_file column.'
    features['subject'] = features[candidates[0]].astype(str).map(lambda s: Path(s).name[:6])
features['subject'] = features['subject'].astype(str).str.strip()
assert features['subject'].str.match(r'^(Ga|Ju|Si)(Co|Pt)\d{2}$').all(), 'Unexpected subject IDs.'
print('Sample feature subjects:', features['subject'].head().tolist())
print('Sample metadata subjects:', metadata['subject'].head().tolist())

# many-to-one mapping because a patient may have multiple walk records
merged = features.merge(metadata, on='subject', how='left', validate='many_to_one', indicator=True,
                        suffixes=('', '_metadata'))
print('Merge outcomes:')
display(merged['_merge'].value_counts().rename_axis('status').reset_index(name='rows'))
assert (merged['_merge'] == 'both').all(), 'Some feature subjects have no matching metadata.'

# keep patients with a valid motor-severity target only 
merged['UPDRSM'] = pd.to_numeric(merged['UPDRSM'], errors='coerce')
patients = merged.loc[(merged['subject'].str[2:4] == 'Pt') & merged['UPDRSM'].notna()].copy()

# checkpoints
assert len(patients) > 0, 'No patients with a valid UPDRSM target.'
assert patients.groupby('subject')['UPDRSM'].nunique().max() == 1, 'Conflicting UPDRSM labels for same subject.'
print('Patient walks:', len(patients), '| Unique patients:', patients['subject'].nunique())
print('UPDRSM range:', patients['UPDRSM'].min(), 'to', patients['UPDRSM'].max())
display(patients[['subject', 'UPDRSM']].head(10))

for below code: need to check features that they match

In [ ]:
# select only numerical gait features.
# exclude metadata, labels, IDs, speed and other clinical scores to avoid target leakage.
metadata_columns = set(metadata.columns) | {'subject', '_merge', 'file', 'filename', 'walk_file',
    'walk', 'study', 'group', 'label', 'target', 'UPDRS', 'UPDRSM', 'HoehnYahr', 'TUAG',
    'Speed_01 (m/sec)', 'Speed_10 (m/sec)', 'Speed_01', 'Speed_10'}

# check for numerical feature columns not in the exclusion set 
feature_columns = [c for c in features.columns if c not in metadata_columns
                   and c in patients.columns and pd.api.types.is_numeric_dtype(patients[c])]
print('Selected features:', feature_columns)
assert feature_columns, 'No numeric feature columns found; inspect features.csv column names.'
assert 'UPDRSM' not in feature_columns, 'Target leakage!' # make sure predictor not in features 

# create feature matrix 
X = patients[feature_columns].replace([np.inf, -np.inf], np.nan) 

# target vector 
y = patients['UPDRSM'].astype(float)

# checkpoints
assert y.notna().all() and np.isfinite(y).all(), 'Invalid target values.'
print('Feature matrix shape:', X.shape, '| Target shape:', y.shape)
print('Missing values by feature:')
display(X.isna().sum().sort_values(ascending=False).head(10).to_frame('missing'))
display(X.head(5))

Setting up model training and model performance helper functions to use for each model. 


!!work in progress!!

In [ ]:
from src.splits import make_subject_folds  # Split using patient IDs, not individual walks.
from src.evaluate import regression_metrics  # Shared MAE/RMSE calculator.
from src.models.linreg import make_linear_regression  # MLR pipeline factory.
from src.models.rf import make_random_forest  # Random forest pipeline factory.
from sklearn.dummy import DummyRegressor  # Training-mean baseline.

folds = make_subject_folds(patients, subject_col='subject', n_splits=5)
assert len(folds) == 5
for fold_number, (train_idx, test_idx) in enumerate(folds, start=1):
    train_ids = set(patients.iloc[train_idx]['subject'])
    test_ids = set(patients.iloc[test_idx]['subject'])
    assert train_ids.isdisjoint(test_ids), 'Subject leakage!'
    print(f'Fold {fold_number}: {len(train_ids)} train subjects / {len(test_ids)} test subjects | overlap = {len(train_ids & test_ids)}')

# Store fold-level results separately for each model; combine only for comparison.
results_by_model = {}
metrics_by_model = {}

def train_classical_model(model_name, model_factory):
    # Helper centralises ONLY the repeated training/saving mechanics.
    # Separate notebook sections below still run and inspect each model independently.
    fold_tables, fold_scores = [], []
    for fold_number, (train_idx, test_idx) in enumerate(folds, start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]  # Gait features for this fold.
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]  # UPDRSM targets for this fold.
        model = model_factory()  # New unfitted model prevents fold-to-fold contamination.
        model.fit(X_train, y_train)  # Fit preprocessing/model on TRAINING data only.
        y_pred = np.asarray(model.predict(X_test)).reshape(-1)  # Predict held-out walks.
        assert len(y_pred) == len(y_test) and np.isfinite(y_pred).all()
        scores = regression_metrics(y_test, y_pred)  # Evaluate on held-out walks.
        fold_scores.append({'model': model_name, 'fold': fold_number, **scores})
        table = pd.DataFrame({'model': model_name, 'fold': fold_number,
                              'subject': patients.iloc[test_idx]['subject'].to_numpy(),
                              'actual_UPDRSM': y_test.to_numpy(), 'predicted_UPDRSM': y_pred})
        fold_tables.append(table)
        prediction_path = OUT / 'predictions' / f'{model_name}_fold{fold_number}.csv'
        weights_path = OUT / 'weights' / f'{model_name}_fold{fold_number}.joblib'
        table.to_csv(prediction_path, index=False)  # Save test predictions for this fold.
        joblib.dump(model, weights_path)  # Save fitted preprocessing and model.
        print(f'{model_name} fold {fold_number}: train={len(train_idx)} test={len(test_idx)} '
              f'MAE={scores["MAE"]:.3f}, RMSE={scores["RMSE"]:.3f} | saved={prediction_path.exists() and weights_path.exists()}')
        display(table.head(3))  # Inspect actual vs predicted values.
    results_by_model[model_name] = pd.concat(fold_tables, ignore_index=True)
    metrics_by_model[model_name] = pd.DataFrame(fold_scores)
    return results_by_model[model_name], metrics_by_model[model_name]

def show_regression_performance(model_name):
    # Use only out-of-fold predictions (every test walk appears once per model).
    table = results_by_model[model_name]
    fold_metrics = metrics_by_model[model_name]
    assert len(table) == len(patients), 'Not all patient walks were evaluated exactly once.'
    assert table.index.size == len(table)
    print(f'{model_name}: per-fold metrics')
    display(fold_metrics)
    scores = regression_metrics(table['actual_UPDRSM'], table['predicted_UPDRSM'])
    print('Pooled out-of-fold MAE/RMSE:', scores)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    actual = table['actual_UPDRSM']; predicted = table['predicted_UPDRSM']
    axes[0].scatter(actual, predicted, alpha=0.65)
    low = min(actual.min(), predicted.min()); high = max(actual.max(), predicted.max())
    axes[0].plot([low, high], [low, high], '--')  # Perfect predictions lie on this line.
    axes[0].set(xlabel='Actual UPDRSM', ylabel='Predicted UPDRSM', title=f'{model_name}: predictions')
    axes[1].scatter(predicted, actual - predicted, alpha=0.65)  # Residual = actual - predicted.
    axes[1].axhline(0, linestyle='--')
    axes[1].set(xlabel='Predicted UPDRSM', ylabel='Residual', title='Residuals')
    fig.tight_layout()
    fig.savefig(OUT / 'figures' / f'{model_name}_diagnostics.png', dpi=160)
    plt.show()
    return scores


### Mean baseline 
This is used as a reference point. Ideally, our models should perform better than this.

In [ ]:
baseline_predictions, baseline_metrics = train_classical_model(
    'mean_baseline', lambda: DummyRegressor(strategy='mean'))
print('Baseline model check:')
display(baseline_metrics)
baseline_scores = show_regression_performance('mean_baseline')

## 2. Model 1: Multiple Linear Regression

In [ ]:
# CHECK COLUMNS: Feature extraction must supply at least two numeric gait predictors.
print('Number of predictors:', X.shape[1])  # One column = one input feature.
print('Number of walking trials:', X.shape[0])  # One row = one observation.
print('Feature names:', list(X.columns))  # Inspect for IDs, UPDRSM or other leakage.
display(X.head())  # Preview the actual values fed to the regression.
assert X.shape[1] >= 2, 'MLR requires at least two predictor columns.'
assert 'UPDRSM' not in X.columns, 'Do not use UPDRSM as an input feature.'
print('MLR input check passed:', X.shape[1] >= 2 and 'UPDRSM' not in X.columns)

In [ ]:
mlr_predictions, mlr_metrics = train_classical_model('linear_regression', make_linear_regression)
print('MLR trained folds:', mlr_metrics['fold'].tolist())

Performance of MLR:

In [ ]:
mlr_scores = show_regression_performance('linear_regression')
# Load one fold's fitted pipeline to inspect its regression coefficients.
mlr_fold1 = joblib.load(OUT / 'weights' / 'linear_regression_fold1.joblib')
coefs = pd.DataFrame({'feature': feature_columns,
                      'standardised_coefficient': mlr_fold1.named_steps['model'].coef_})
coefs['absolute_coefficient'] = coefs['standardised_coefficient'].abs()

# outputs 
print('MLR intercept (fold 1):', mlr_fold1.named_steps['model'].intercept_)
display(coefs.sort_values('absolute_coefficient', ascending=False).head(15))

# checkpoints
assert len(coefs) == X.shape[1], 'Coefficient count does not match feature count.' 

## Model 2: Random Forest